In [44]:
import requests
import pandas as pd
from tqdm import tqdm
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

In [45]:
# ================= CONFIG =================
INPUT_FILES = [
    "north_of_vietnam.csv",
    "central_of_vietnam.csv",
    "south_of_vietnam.csv"
]

START_DATE = "2025-06-01"
END_DATE = "2025-09-30"

OUTPUT_FILE = "aqi_air_weather_6_9_2025.csv"
MAX_THREADS = 10

In [46]:
# ================= LOAD =================
dfs = [pd.read_csv(f) for f in INPUT_FILES]
df = pd.concat(dfs, ignore_index=True)

In [51]:
# ================= API =================

def get_air_quality(lat, lon):
    url = "https://air-quality-api.open-meteo.com/v1/air-quality"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": START_DATE,
        "end_date": END_DATE,
        "hourly": [
            "pm2_5",
            "pm10",
            "carbon_monoxide",
            "nitrogen_dioxide",
            "us_aqi"
        ],
        "timezone": "Asia/Bangkok"
    }

    try:
        res = requests.get(url, params=params, timeout=30)
        if res.status_code == 200:
            return res.json().get("hourly", None)
    except:
        pass
    return None


def get_weather(lat, lon):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": START_DATE,
        "end_date": END_DATE,
        "hourly": [
            "temperature_2m",
            "relative_humidity_2m",
            "windspeed_10m"
        ],
        "timezone": "Asia/Bangkok"
    }

    try:
        res = requests.get(url, params=params, timeout=30)
        if res.status_code == 200:
            # Corrected: Access 'hourly' data instead of 'daily'
            return res.json().get("hourly", None)
    except:
        pass
    return None

In [52]:
# ================= PROCESS =================
def process_location(row):
    lat = row["lat"]
    lon = row["lon"]
    district = row.get("district", "")
    city = row.get("city", "")

    try:
        # ===== AQI HOURLY =====
        air = get_air_quality(lat, lon)
        if air is None:
            return []

        air_df = pd.DataFrame({
            "datetime": air["time"],
            "pm2_5": air["pm2_5"],
            "pm10": air["pm10"],
            "co": air["carbon_monoxide"],
            "no2": air["nitrogen_dioxide"],
            "aqi": air["us_aqi"]
        })

        air_df["datetime"] = pd.to_datetime(air_df["datetime"])

        # ===== WEATHER HOURLY  =====
        weather = get_weather(lat, lon)
        if weather is None:
            return []

        # Corrected: Use 'datetime' for hourly weather data
        weather_df = pd.DataFrame({
            "datetime": weather["time"],
            "temp": weather["temperature_2m"],
            "humidity": weather["relative_humidity_2m"],
            "wind": weather["windspeed_10m"]
        })

        weather_df["datetime"] = pd.to_datetime(weather_df["datetime"])

        # ===== MERGE theo DATETIME =====
        merged = pd.merge(air_df, weather_df, on="datetime", how="inner")

        merged["district"] = district
        merged["city"] = city
        merged["lat"] = lat
        merged["lon"] = lon

        return merged.to_dict("records")

    except Exception as e:
        print(f"Lỗi {district}-{city}:", e)
        return []

In [54]:
# ================= MULTI THREAD =================
results = []

with ThreadPoolExecutor(max_workers=MAX_THREADS) as executor:
    futures = [executor.submit(process_location, row) for _, row in df.iterrows()]

    for future in tqdm(as_completed(futures), total=len(futures)):
        results.extend(future.result())

100%|██████████| 686/686 [00:49<00:00, 13.83it/s]


In [55]:
# ================= SAVE =================
output_df = pd.DataFrame(results)

output_df = output_df.sort_values(by=["city", "district", "datetime"])

output_df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print("✅ DONE:", OUTPUT_FILE)

✅ DONE: aqi_air_weather_6_9_2025.csv


In [56]:
df = pd.read_csv("/content/aqi_air_weather_6_9_2025.csv")

In [57]:
df.head()

,datetime,pm2_5,pm10,co,no2,aqi,temp,humidity,wind,district,city,lat,lon
0,2025-06-01 00:00:00,74.3,77.5,193.0,33.5,103,26.7,89,9.6,An Phu,An Giang,10.9167,105.0833
1,2025-06-01 01:00:00,71.9,75.4,181.0,32.1,107,26.9,90,7.7,An Phu,An Giang,10.9167,105.0833
2,2025-06-01 02:00:00,63.8,67.1,170.0,30.4,111,26.6,92,5.1,An Phu,An Giang,10.9167,105.0833
3,2025-06-01 03:00:00,57.4,60.8,161.0,28.3,114,26.4,93,4.2,An Phu,An Giang,10.9167,105.0833
4,2025-06-01 04:00:00,57.2,60.5,161.0,26.0,116,26.5,93,7.0,An Phu,An Giang,10.9167,105.0833


In [58]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 617808 entries, 0 to 617807
Data columns (total 13 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   datetime  617808 non-null  object 
 1   pm2_5     617808 non-null  float64
 2   pm10      617808 non-null  float64
 3   co        617808 non-null  float64
 4   no2       617808 non-null  float64
 5   aqi       617808 non-null  int64  
 6   temp      617808 non-null  float64
 7   humidity  617808 non-null  int64  
 8   wind      617808 non-null  float64
 9   district  617808 non-null  object 
 10  city      617808 non-null  object 
 11  lat       617808 non-null  float64
 12  lon       617808 non-null  float64
dtypes: float64(8), int64(2), object(3)
memory usage: 61.3+ MB
